# Мерей: воспроизведение оптимизации при пике до 827 MiB

Неисполненный шаблон: результатов и заранее заполненных outputs нет.
В конфигурации укажите измеренный `WINNER_COMMIT`. Базовый малопамятный
вариант V5r1 закреплён коммитом `31d02f87b577395cffc7cabf54b311ae83761097`.

Два блока ABBA дают восемь полных запусков: четыре V5r1 и четыре выбранного
решения. Сохраняются отдельно полное время команды, все 64 шага и шаги 3–64.
Пик каждого малопамятного запуска должен быть не выше 827 MiB.
Другая серия исходного official harness выполняет два повтора, quality eval
и все десять правил ТЗ. Результаты двух серий сохраняются раздельно.

Kaggle: T4 x2, Internet On, новая сессия, затем Run All. Исходный harness
использует одну T4 и сам измеряет полный внешний пик NVML/nvidia-smi.
Все GPU children запускаются последовательно; контроллер использует только
стандартную библиотеку и не импортирует Torch.

До измерений закрепляются зависимости и сверяются все Git blobs решения,
неизменность harness/task/tests, исходные данные и реальная инициализация
Soup. Для проверки инициализации отдельно загружается прежний быстрый Мерей
с полностью резидентными весами. Его время здесь не измеряется.

Собственный CPU-offload и пересчёт выполняются внутри solution; модель,
NF4, LoRA, dropout, seed, batch/order, 64 шага и настройки Soup определяет
исходный task. Шаблон не переключает параметры обучения.


In [ ]:
import hashlib, importlib.metadata as metadata, json, os
import shutil, statistics, subprocess, sys, zipfile
from pathlib import Path

REPO_URL = 'https://github.com/promaladoi/Decentrathon-6.0.git'
WINNER_COMMIT = '28f8a3426582444fb493e74574caf74eea719c9b'
LOW_MEMORY_BASELINE = '31d02f87b577395cffc7cabf54b311ae83761097'
FAST_REFERENCE = 'bcbe3e82e1c9483b8d24b77a166e39b9cd787b0e'
CANDIDATE, PREVIOUS_MEREI = WINNER_COMMIT, LOW_MEMORY_BASELINE
ANAIS_SOURCE = '78a2ec998ca7b665d1aa70299aeb6f2f2a105a57'
assert len(CANDIDATE) == 40 and all(c in '0123456789abcdef' for c in CANDIDATE), 'Fill WINNER_COMMIT'
MEMORY_CEILING_MIB = 827
ABBA_ROUNDS = 2
assert type(ABBA_ROUNDS) is int and ABBA_ROUNDS >= 2
SOURCE_COMMITS = {'candidate': CANDIDATE, 'previous': PREVIOUS_MEREI,
                  'fast_reference': FAST_REFERENCE}
MEASURED_LABELS = ('previous', 'candidate')
WORKING = Path('/kaggle/working')
EVIDENCE = WORKING / ('merey-memory827-' + CANDIDATE[:12] + '-evidence')
EVIDENCE.mkdir(exist_ok=False)
gpu_names = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True)
print(gpu_names, flush=True)
assert 'T4' in gpu_names, 'Choose GPU T4 x2'


In [ ]:
def installed(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

# The controller uses distribution metadata, not imports of these libraries.
# Finish every install/removal here before any test or training child starts.
assert installed('torch') == '2.11.0+cu128', 'Use the recorded Kaggle image'
assert installed('triton') == '3.6.0', 'Use Triton 3.6.0 from the recorded image'
requirements = {
    'soup-cli': ('0.77.0', 'soup-cli[train]==0.77.0'),
    'transformers': ('5.16.1', 'transformers==5.16.1'),
    'peft': ('0.20.0', 'peft==0.20.0'),
    'trl': ('0.29.1', 'trl==0.29.1'),
    'bitsandbytes': ('0.50.2', 'bitsandbytes==0.50.2'),
    'accelerate': ('1.14.0', 'accelerate==1.14.0'),
    'datasets': ('4.8.5', 'datasets==4.8.5'),
}
missing = [spec for name, (required, spec) in requirements.items()
           if installed(name) != required]
if installed('pytest') is None:
    missing.append('pytest')
if installed('torchao') is not None:
    # Remove the optional incompatible package before the fixed environment begins.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
versions = {name: installed(name) for name in ['torch', 'triton', *requirements, 'pytest']}
assert all(versions[name] == required for name, (required, spec) in requirements.items())
assert versions['torch'] == '2.11.0+cu128' and versions['triton'] == '3.6.0'
print(json.dumps(versions, indent=2), flush=True)
# No later cell installs, removes, upgrades or edits packages.

PACKAGES_FIXED = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(EVIDENCE / 'fixed-packages.txt').write_text(PACKAGES_FIXED, encoding='utf-8')


In [ ]:
def clone_exact(label, commit):
    checkout = WORKING / ('merey-memory827-' + label + '-' + commit[:12])
    assert not checkout.exists(), 'Preserve previous results and use a fresh session'
    subprocess.run(['git', '-c', 'core.autocrlf=false', 'clone', '-q', '--no-checkout', REPO_URL, str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'config', 'core.autocrlf', 'false'], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', '--detach', commit], check=True)
    actual = subprocess.check_output(
        ['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip()
    assert actual == commit
    return checkout / 'Мерей'

roots = {label: clone_exact(label, commit) for label, commit in SOURCE_COMMITS.items()}
for directory in ['harness', 'task', 'tests']:
    a, b = roots['previous'] / directory, roots['candidate'] / directory
    files = sorted(p.relative_to(a) for p in a.rglob('*') if p.is_file())
    assert files == sorted(p.relative_to(b) for p in b.rglob('*') if p.is_file())
    assert all((a / p).read_bytes() == (b / p).read_bytes() for p in files), directory

sys.path.insert(0, str(roots['candidate'] / 'harness'))
from common import ENV
assert ENV['CUDA_VISIBLE_DEVICES'] == '0', 'Use the original one-GPU harness environment'
assert ENV['PYTORCH_CUDA_ALLOC_CONF'] == 'expandable_segments:True'

def run_repo(label, arguments, log_name, **overrides):
    print('RUN_BEGIN', label, arguments, flush=True)
    log = EVIDENCE / log_name
    with log.open('w', encoding='utf-8') as stream:
        result = subprocess.run([sys.executable, *arguments], cwd=roots[label],
                                env=dict(ENV, **overrides), stdout=stream,
                                stderr=subprocess.STDOUT)
    output = log.read_text(encoding='utf-8')
    print(output, flush=True)
    result.check_returncode()
    return output

def solution_hash(root, python_only=False):
    digest = hashlib.sha256()
    for path in sorted((root / 'solution').rglob('*')):
        if (path.is_file() and '__pycache__' not in path.parts
                and (not python_only or path.suffix == '.py')):
            digest.update(path.relative_to(root / 'solution').as_posix().encode())
            digest.update(path.read_bytes())
    return digest.hexdigest()

measured_solution_git_tree = subprocess.check_output(
    ['git', '-C', str(roots['candidate'].parent), 'rev-parse',
     CANDIDATE + ':Мерей/solution'], text=True).strip()
assert len(measured_solution_git_tree) == 40

provenance = {
    'candidate_runtime_commit': CANDIDATE,
    'previous_merei_runtime_commit': PREVIOUS_MEREI,
    'fast_reference_commit': FAST_REFERENCE,
    'versions': versions,
    'candidate_solution_sha256_before_tests': solution_hash(roots['candidate']),
    'candidate_python_sha256': solution_hash(roots['candidate'], python_only=True),
    'measured_solution_git_tree': measured_solution_git_tree,
    'memory_ceiling_mib': MEMORY_CEILING_MIB,
    'anais_source_commit': ANAIS_SOURCE,
    'anais_source_path': 'Анаис/solution',
}
provenance['previous_solution_sha256'] = solution_hash(roots['previous'])
(EVIDENCE / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
print(json.dumps(provenance, indent=2), flush=True)

for directory in ('harness', 'task', 'tests'):
    reference_tree = subprocess.check_output(['git', '-C', str(roots['previous'].parent),
        'rev-parse', PREVIOUS_MEREI + ':Мерей/' + directory], text=True).strip()
    for label, commit in SOURCE_COMMITS.items():
        actual_tree = subprocess.check_output(['git', '-C', str(roots[label].parent),
            'rev-parse', commit + ':Мерей/' + directory], text=True).strip()
        assert actual_tree == reference_tree, (directory, label)


In [ ]:
def verify_solution_git_blobs(root, commit):
    tree = subprocess.check_output(
        ['git', '-C', str(root.parent), 'rev-parse', commit + ':Мерей/solution'], text=True).strip()
    listing = subprocess.check_output(['git', '-C', str(root.parent), 'ls-tree', '--full-tree', '-rz', tree])
    records = []
    for entry in listing.split(b'\0'):
        if not entry:
            continue
        descriptor, raw_path = entry.split(b'\t', 1)
        mode, kind, expected = descriptor.decode().split()
        assert kind == 'blob' and mode == '100644'
        path = raw_path.decode('utf-8')
        raw = (root / 'solution' / path).read_bytes()
        actual = hashlib.sha1(b'blob ' + str(len(raw)).encode() + b'\0' + raw).hexdigest()
        assert actual == expected, 'Checkout bytes differ from Git: ' + path
        records.append({'path': path, 'git_blob_sha': actual,
                        'sha256': hashlib.sha256(raw).hexdigest(), 'bytes': len(raw)})
    assert records
    expected_paths = {record['path'] for record in records}
    actual_paths = {p.relative_to(root / 'solution').as_posix()
                    for p in (root / 'solution').rglob('*')
                    if p.is_file() and '__pycache__' not in p.parts}
    assert actual_paths == expected_paths, 'Unexpected solution files'
    return {'commit': commit, 'solution_git_tree': tree, 'files': records,
            'solution_sha256': solution_hash(root),
            'python_sha256': solution_hash(root, python_only=True)}

source_proofs = {label: verify_solution_git_blobs(roots[label], commit)
                 for label, commit in SOURCE_COMMITS.items()}
for directory in ['harness', 'task', 'tests']:
    trees = [subprocess.check_output(
        ['git', '-C', str(roots[label].parent), 'rev-parse', commit + ':Мерей/' + directory],
        text=True).strip() for label, commit in [('previous', PREVIOUS_MEREI), ('candidate', CANDIDATE)]]
    assert trees[0] == trees[1], directory + ' must remain unchanged'
provenance['source_proofs'] = source_proofs
(EVIDENCE / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')


## Native parity, tests и исходные данные

CPU diagnostic вызывает настоящий Soup setup-prefix и доказывает native
metadata parity, восстановление функции и отсутствие model/seed/CUDA work.
Только его отдельный child получает пустой CUDA_VISIBLE_DEVICES.
Затем исполняются полный solution suite и все 79 Soup regressions с решением.
Counts solution suite не зашиты: фактические passed/skipped сохраняются в логах.
task/prepare_data.py запускается без изменений, bytes данных сравниваются.


In [ ]:
parity_output = run_repo(
    'candidate', ['solution/test_package_metadata_snapshot.py', 'NativeInstalledTests'],
    'native-metadata-parity.log',
    MEREY_NATIVE_METADATA_SNAPSHOT_PARITY='1', CUDA_VISIBLE_DEVICES='')
prefix = 'native metadata snapshot parity: '
records = [json.loads(line.split(prefix, 1)[1])
           for line in parity_output.splitlines() if prefix in line]
assert len(records) == 1, 'The explicit installed-native parity diagnostic must run'
parity = records[0]
assert parity['native_mapping_equal'] and parity['independent_dict_and_lists']
assert parity['function_restored'] and not parity['cuda_initialized']
assert parity['cpu_probe_gpu_hidden']
assert parity['setup_calls'] == {'wrapper': 1, 'native': 1}
assert parity['actual_snapshot_returns_to_trl'] == 1
assert parity['native_scan_callers_during_setup'] == []
assert parity['stopped_before_seed_and_model'] and parity['setup_idempotent']
(EVIDENCE / 'native-metadata-parity.json').write_text(
    json.dumps(parity, indent=2), encoding='utf-8')


In [ ]:
run_repo('candidate', ['-m', 'pytest', 'solution', '-q', '-p', 'no:cacheprovider'],
         'solution-tests.log')
run_repo('candidate', ['-m', 'pytest', 'tests', '-q', '-p', 'no:cacheprovider'],
         'soup-regressions.log', SOUP_APPLY_SOLUTION='1')
for label in roots:
    run_repo(label, ['task/prepare_data.py'], label + '-prepare-data.log')
assert solution_hash(roots['candidate']) == provenance['candidate_solution_sha256_before_tests']

assert '79 passed' in (EVIDENCE / 'soup-regressions.log').read_text(encoding='utf-8')
for filename in ['train.jsonl', 'eval_public.jsonl']:
    previous = roots['previous'] / 'data' / filename
    candidate = roots['candidate'] / 'data' / filename
    assert previous.read_bytes() == candidate.read_bytes(), 'Task data/order must match'
    destination = EVIDENCE / 'task-data' / filename
    destination.parent.mkdir(exist_ok=True)
    shutil.copyfile(candidate, destination)
for label in roots:
    assert solution_hash(roots[label]) == source_proofs[label]['solution_sha256']

for filename in ('train.jsonl', 'eval_public.jsonl'):
    assert (roots['fast_reference'] / 'data' / filename).read_bytes() == (roots['candidate'] / 'data' / filename).read_bytes()


In [ ]:
warmup_child = "\n".join([
    "import sys", "from pathlib import Path",
    "sys.path.insert(0, str(Path.cwd() / 'harness'))",
    "from run_task import warm_up",
    "out = Path(sys.argv[1]); out.mkdir(parents=True, exist_ok=False)",
    "warm_up(out)", ""])
for label in MEASURED_LABELS:
    run_repo(label, ['-c', warmup_child, 'runs_memory827_ab_warmup'], 'ab-' + label + '-warmup.log')



## Реальная инициализация Soup

Два свежих процесса вызывают настоящий Soup CLI с исходным config builder:
прежнее решение E и выбранный малопамятный вариант. Проверяются все
NF4-байты/scales, остальные frozen values, LoRA, настройки и CPU/CUDA RNG
после native PEFT initialization. Диагностический observer останавливает
процесс до trainer; эта серия не считается измерением обучения.


In [ ]:
INITIALIZATION_PARITY_SOURCE = '"""Independent native Soup initialization parity, not a training benchmark.\n\nThe stdlib controller starts two fresh serial children using the original\nharness runner/environment/config builder. Each child calls the real Soup CLI,\nafter the same CUDA context initialization and solution.apply() boundary as\nharness/_train.py. An outer get_peft_model observer captures the returned final\nPEFT model before trainer/tokenization, then stops via a private BaseException.\nNo production code or task/harness file is changed.\n"""\nimport argparse\nimport functools\nimport hashlib\nimport json\nfrom pathlib import Path\nimport pickle\nimport random\nimport sys\nimport traceback\n\n\nKIND = \'memory900_initialization_parity_not_training_admission\'\n\n\ndef _json_default(value):\n    # PEFT uses an unordered set for target_modules. Its repr is process/hash-\n    # seed dependent, whereas the adapter setting itself is order independent.\n    if isinstance(value, (set, frozenset)):\n        return sorted(value)\n    return str(value)\n\n\ndef _json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=True, default=_json_default),\n                          encoding=\'utf-8\')\n\n\ndef child(repo, config, output):\n    repo, config, output = Path(repo).resolve(), Path(config).resolve(), Path(output).resolve()\n    sys.path.insert(0, str(repo / \'harness\'))\n    sys.argv = [\'soup\', \'train\', \'--config\', str(config), \'--yes\']\n    sys.modules[\'unsloth\'] = None\n    sys.modules[\'unsloth_zoo\'] = None\n    import torch\n    torch.empty(1, device=\'cuda\')\n    import transformers\n    from _train import TritonWatch\n    watch = TritonWatch(torch)\n    sys.path.insert(0, str(repo))\n    import solution\n    solution.apply()\n\n    # Soup imports this symbol inside _setup_transformers after apply(). This\n    # outer wrapper observes the result of any existing solution wrapper too.\n    import peft\n    import bitsandbytes as bnb\n    import numpy as np\n    native_get_peft_model = peft.get_peft_model\n    captured = []\n\n    class CapturedInitialization(BaseException):\n        pass\n\n    def tensor_digest(tensor):\n        detached = tensor.detach().contiguous().cpu()\n        raw = detached.reshape(-1).view(torch.uint8).numpy().tobytes()\n        return {\'shape\': list(tensor.shape), \'dtype\': str(tensor.dtype),\n                \'sha256\': hashlib.sha256(raw).hexdigest(), \'bytes\': len(raw),\n                \'device\': str(tensor.device)}\n\n    def canonical_frozen_digest(tensor):\n        # The current solution proves the tied F32 embedding values are exactly\n        # representable as F16. Compare actual F32 values without allocating a\n        # whole extra embedding on either GPU or CPU at once.\n        digest = hashlib.sha256()\n        flat = tensor.detach().reshape(-1)\n        for begin in range(0, flat.numel(), 1024 * 1024):\n            values = flat[begin:begin + 1024 * 1024].cpu().to(torch.float32)\n            digest.update(values.view(torch.uint8).numpy().tobytes())\n        return {\'shape\': list(tensor.shape), \'canonical_dtype\': \'torch.float32\',\n                \'sha256\': digest.hexdigest(), \'elements\': tensor.numel(),\n                \'storage_dtype\': str(tensor.dtype), \'device\': str(tensor.device)}\n\n    def quant_state(state):\n        result = {\'shape\': list(state.shape) if state.shape is not None else None,\n                  \'dtype\': str(state.dtype), \'blocksize\': state.blocksize,\n                  \'quant_type\': state.quant_type, \'nested\': bool(state.nested),\n                  \'tensors\': {}}\n        for name in (\'absmax\', \'code\', \'offset\'):\n            value = getattr(state, name, None)\n            if isinstance(value, torch.Tensor):\n                result[\'tensors\'][name] = tensor_digest(value)\n            elif value is not None:\n                result[name] = value\n        second = getattr(state, \'state2\', None)\n        if second is not None:\n            result[\'state2\'] = quant_state(second)\n        return result\n\n    def rng_digest():\n        return {\'torch_cpu\': tensor_digest(torch.get_rng_state()),\n                \'torch_cuda\': [tensor_digest(state) for state in torch.cuda.get_rng_state_all()],\n                \'python_sha256\': hashlib.sha256(pickle.dumps(random.getstate(), protocol=4)).hexdigest(),\n                \'numpy_sha256\': hashlib.sha256(pickle.dumps(np.random.get_state(), protocol=4)).hexdigest()}\n\n    @functools.wraps(native_get_peft_model)\n    def capture(*args, **kwargs):\n        model = native_get_peft_model(*args, **kwargs)\n        if captured:\n            raise RuntimeError(\'unexpected second get_peft_model call before capture stop\')\n        snapshot = {\'kind\': KIND, \'stage\': \'after_get_peft_model_before_trainer\',\n                    \'rng\': rng_digest(), \'trainable\': {}, \'frozen_nf4\': {}, \'other_frozen\': {}}\n        # Diagnostic metadata is outside all compared numeric/config fields.\n        # Loading an earlier no-frame solution remains a valid comparison.\n        frame_module = sys.modules.get(\'solution.nf4_transfer_frame\')\n        frame_checker = getattr(frame_module, \'frame_is_current\', None)\n        frames = {\'nf4_parameters\': 0, \'bound_frames\': 0, \'current_frames\': 0,\n                  \'checker_loaded\': callable(frame_checker), \'unbound_names\': [],\n                  \'bound_but_invalid_names\': []}\n        for name, parameter in model.named_parameters():\n            if parameter.requires_grad:\n                snapshot[\'trainable\'][name] = tensor_digest(parameter)\n            elif isinstance(parameter, bnb.nn.Params4bit):\n                frames[\'nf4_parameters\'] += 1\n                state = parameter.quant_state\n                if getattr(state, \'_solution_transfer_frame\', None) is not None:\n                    frames[\'bound_frames\'] += 1\n                    if callable(frame_checker) and frame_checker(parameter, state):\n                        frames[\'current_frames\'] += 1\n                    else:\n                        frames[\'bound_but_invalid_names\'].append(name)\n                else:\n                    frames[\'unbound_names\'].append(name)\n                snapshot[\'frozen_nf4\'][name] = {\n                    \'packed\': tensor_digest(parameter),\n                    \'quant_state\': quant_state(parameter.quant_state)}\n            else:\n                snapshot[\'other_frozen\'][name] = canonical_frozen_digest(parameter)\n        snapshot[\'trainable_elements\'] = sum(p.numel() for p in model.parameters() if p.requires_grad)\n        snapshot[\'trainable_count\'] = len(snapshot[\'trainable\'])\n        snapshot[\'frozen_nf4_count\'] = len(snapshot[\'frozen_nf4\'])\n        frames[\'all_nf4_frames_current\'] = (\n            frames[\'nf4_parameters\'] > 0\n            and frames[\'current_frames\'] == frames[\'nf4_parameters\'])\n        snapshot[\'frame_diagnostics\'] = frames\n        snapshot[\'adapter_config\'] = {name: value.to_dict() for name, value in model.peft_config.items()}\n        snapshot[\'source\'] = {\'repo\': str(repo), \'solution_apply_module\': solution.__file__}\n        snapshot[\'observer_calls\'] = 1\n        snapshot[\'stopped_before_trainer\'] = True\n        captured.append(snapshot)\n        _json(output, snapshot)\n        print(json.dumps({\'marker\': \'INITIAL_PARITY_CAPTURED\', \'kind\': KIND,\n                          \'trainable_count\': snapshot[\'trainable_count\'],\n                          \'trainable_elements\': snapshot[\'trainable_elements\'],\n                          \'frozen_nf4_count\': snapshot[\'frozen_nf4_count\']}), flush=True)\n        raise CapturedInitialization()\n\n    peft.get_peft_model = capture\n    try:\n        from soup_cli.cli import run\n        run()\n    except CapturedInitialization:\n        if len(captured) != 1:\n            raise RuntimeError(\'capture count differs from one\')\n    except BaseException as exc:\n        if not captured:\n            _json(output, {\'kind\': KIND, \'error\': repr(exc), \'traceback\': traceback.format_exc(),\n                           \'observer_calls\': 0, \'stopped_before_trainer\': False})\n        raise\n    else:\n        raise RuntimeError(\'Soup returned without the PEFT initialization capture\')\n    finally:\n        peft.get_peft_model = native_get_peft_model\n    # The original warm-up thread compiles without RNG or training arithmetic.\n    # Let it finish before the fresh process exits; never interrupt that thread.\n    from solution import warmup\n    warmup.wait()\n    assert watch.error is None, watch.error\n\n\ndef _without_placement(value):\n    if isinstance(value, dict):\n        return {key: _without_placement(item) for key, item in value.items()\n                if key not in (\'device\', \'storage_dtype\')}\n    if isinstance(value, list):\n        return [_without_placement(item) for item in value]\n    return value\n\n\ndef _first_mismatches(left, right, prefix=\'\', result=None):\n    result = [] if result is None else result\n    if len(result) >= 20:\n        return result\n    if isinstance(left, dict) and isinstance(right, dict):\n        for key in sorted(set(left) | set(right)):\n            path = prefix + \'.\' + key if prefix else key\n            if key not in left or key not in right:\n                result.append({\'path\': path, \'current_present\': key in left,\n                               \'candidate_present\': key in right})\n            else:\n                _first_mismatches(left[key], right[key], path, result)\n            if len(result) >= 20:\n                break\n    elif left != right:\n        result.append({\'path\': prefix, \'current\': left, \'candidate\': right})\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--child\', action=\'store_true\')\n    parser.add_argument(\'--repo\', type=Path)\n    parser.add_argument(\'--config\', type=Path)\n    parser.add_argument(\'--capture\', type=Path)\n    parser.add_argument(\'--current-repo\', type=Path)\n    parser.add_argument(\'--candidate-repo\', type=Path)\n    parser.add_argument(\'--train-file\', type=Path)\n    parser.add_argument(\'--out\', type=Path)\n    parser.add_argument(\'--seed\', type=int, default=1)\n    args = parser.parse_args()\n    if args.child:\n        if None in (args.repo, args.config, args.capture):\n            parser.error(\'--repo, --config and --capture required in child mode\')\n        child(args.repo, args.config, args.capture)\n        return\n    if None in (args.current_repo, args.candidate_repo, args.train_file, args.out):\n        parser.error(\'--current-repo, --candidate-repo, --train-file and --out required\')\n    current, candidate, out = (p.resolve() for p in\n                               (args.current_repo, args.candidate_repo, args.out))\n    assert (current / \'task/task.yaml\').read_bytes() == (candidate / \'task/task.yaml\').read_bytes()\n    sys.path.insert(0, str(current / \'harness\'))\n    from common import OFFLINE_ENV, run_logged\n    from run_task import make_config\n    out.mkdir(parents=True, exist_ok=True)\n    rows = {}\n    for name, repo in ((\'current\', current), (\'candidate\', candidate)):\n        run_dir = out / name\n        run_dir.mkdir(parents=True, exist_ok=False)\n        config = make_config(run_dir, args.train_file.resolve(), args.seed)\n        capture_path = run_dir / \'initialization.json\'\n        env = dict(OFFLINE_ENV, SOUP_DB_PATH=str(run_dir / \'experiments.db\'),\n                   TRITON_CACHE_DIR=str(run_dir / \'triton_cache\'),\n                   TORCHINDUCTOR_CACHE_DIR=str(run_dir / \'inductor_cache\'))\n        command = [sys.executable, str(Path(__file__).resolve()), \'--child\',\n                   \'--repo\', str(repo), \'--config\', str(config), \'--capture\', str(capture_path)]\n        code, wall, peak, start = run_logged(command, run_dir / \'initialization.log\', 900, env)\n        rows[name] = {\'exit_code\': code, \'diagnostic_wall_s\': wall,\n                      \'diagnostic_gpu_peak_mib\': peak, \'gpu_start_mib\': start}\n        if code != 0 or not capture_path.exists():\n            _json(out / \'summary.json\', {\'kind\': KIND, \'runs\': rows, \'all_equal\': False,\n                                        \'error\': name + \' initialization capture failed\'})\n            raise RuntimeError(f\'{name} initialization capture failed; inspect {run_dir}\')\n    left = json.loads((out / \'current/initialization.json\').read_text())\n    right = json.loads((out / \'candidate/initialization.json\').read_text())\n    fields = (\'trainable\', \'rng\', \'frozen_nf4\', \'other_frozen\', \'adapter_config\',\n              \'trainable_elements\', \'trainable_count\', \'frozen_nf4_count\',\n              \'observer_calls\', \'stopped_before_trainer\', \'stage\')\n    checks = {key: _without_placement(left[key]) == _without_placement(right[key]) for key in fields}\n    summary = {\'kind\': KIND, \'runs\': rows, \'seed\': args.seed, \'equal\': checks,\n               \'all_equal\': all(checks.values()),\n               \'frame_diagnostics\': {\'current\': left[\'frame_diagnostics\'],\n                                     \'candidate\': right[\'frame_diagnostics\']},\n               \'first_mismatches\': _first_mismatches(\n                   {key: _without_placement(left[key]) for key in fields},\n                   {key: _without_placement(right[key]) for key in fields})}\n    _json(out / \'summary.json\', summary)\n    print(json.dumps(summary, ensure_ascii=True, indent=2), flush=True)\n    if not summary[\'all_equal\']:\n        raise SystemExit(2)\n\n\nif __name__ == \'__main__\':\n    main()\n'
initial_script = EVIDENCE / 'initialization-parity.py'
initial_script.write_text(INITIALIZATION_PARITY_SOURCE, encoding='utf-8')
initial_raw_out = WORKING / ('memory827-initialization-' + CANDIDATE[:12])
run_repo('candidate', [str(initial_script), '--current-repo', str(roots['fast_reference']),
    '--candidate-repo', str(roots['candidate']), '--train-file', str(roots['candidate'] / 'data/train.jsonl'),
    '--out', str(initial_raw_out), '--seed', '1'], 'initialization-parity-controller.log')
initial_parity = json.loads((initial_raw_out / 'summary.json').read_text())
assert initial_parity['all_equal'], initial_parity['first_mismatches']
assert all(initial_parity['equal'].values())
assert all(run['exit_code'] == 0 for run in initial_parity['runs'].values())
for path in initial_raw_out.rglob('*'):
    if (path.is_file() and path.suffix in ('.json', '.log', '.yaml')
            and not any(part in ('triton_cache', 'inductor_cache', '__pycache__') for part in path.parts)):
        destination = EVIDENCE / 'initialization-parity' / path.relative_to(initial_raw_out)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(path, destination)
for label, commit in SOURCE_COMMITS.items():
    assert verify_solution_git_blobs(roots[label], commit) == source_proofs[label]
print('REAL_SOUP_INITIALIZATION_PARITY_COMPLETE', json.dumps(initial_parity), flush=True)


## V5r1 → выбранный вариант: два блока ABBA

Восемь свежих training children используют неизменённый `run_task.train`:
полные 64 шага, seed исходного task, внешний timer, опрос NVML и пустые
compiler caches. Между запусками не запускается дополнительный eval.
Ускорение шагов вычисляется по медианам четырёх запусков каждого решения;
положительный процент означает сокращение времени. Сохраняется разброс.
Каждое значение памяти проверяется против строгой границы 827 MiB.


In [ ]:
training_child = "\n".join([
    "import sys", "from pathlib import Path", "from types import SimpleNamespace",
    "sys.path.insert(0, str(Path.cwd() / 'harness'))",
    "from common import REPO, write_json",
    "from run_task import load_task, train",
    "name = sys.argv[1]",
    "out = REPO / 'runs_memory827_previous_ab'; out.mkdir(exist_ok=True)",
    "task = load_task()",
    "args = SimpleNamespace(out=out, train_file=REPO/'data'/'train.jsonl',",
    "                       eval_file=REPO/'data'/'eval_public.jsonl',",
    "                       seed=task['training']['seed'], timeout=900)",
    "result = train('solution', name, args)",
    "write_json(out / (name + '-result.json'), result)",
    "assert result['exit_code'] == 0",
    "assert result['global_step'] == 64 and result['metrics']['steps'] == 64",
    "assert result['metrics']['timed_steps'] == 62",
    "print('NATIVE_AB_TRAIN_COMPLETE', name, flush=True)", ""])
(EVIDENCE / 'ab-training-controller.py').write_text(training_child, encoding='utf-8')
order = []
for block in range(ABBA_ROUNDS):
    first = 2 * block + 1
    order.extend([('previous', 'old_' + str(first)), ('candidate', 'new_' + str(first)),
                  ('candidate', 'new_' + str(first + 1)), ('previous', 'old_' + str(first + 1))])
assert len(order) == 4 * ABBA_ROUNDS and len({name for label, name in order}) == len(order)
ab_runs = {'previous': [], 'candidate': []}
for label, name in order:
    run_repo(label, ['-c', training_child, name], 'ab-' + name + '.log')
    path = roots[label] / 'runs_memory827_previous_ab' / (name + '-result.json')
    run = json.loads(path.read_text())
    assert run['exit_code'] == 0
    assert run['metrics']['steps'] == 64 and run['metrics']['timed_steps'] == 62
    assert run['gpu_start_mib'] < 1000, 'A/B GPUs must be idle at each run start'
    assert run['metrics']['visible_devices'] == '0' and run['metrics']['device_count'] == 1
    assert run['metrics']['alloc_conf'] == 'expandable_segments:True'
    assert run['gpu_peak_mib'] <= MEMORY_CEILING_MIB, (label, name, 'exceeds 827 MiB')
    ab_runs[label].append(run)
    shutil.copyfile(path, EVIDENCE / ('ab-' + name + '-result.json'))

def summarize_current(rows):
    return {
        'wall_runs_s': [r['wall_s'] for r in rows],
        'wall_median_s': statistics.median(r['wall_s'] for r in rows),
        'gpu_peaks_mib': [r['gpu_peak_mib'] for r in rows],
        'gpu_peak_mib': max(r['gpu_peak_mib'] for r in rows),
        'train_loop_runs_s': [r['metrics']['train_loop_s'] for r in rows],
        'train_loop_median_s': statistics.median(r['metrics']['train_loop_s'] for r in rows),
        'timed_steps_3_to_64_runs_s': [r['metrics']['timed_loop_s'] for r in rows],
        'timed_steps_3_to_64_median_s': statistics.median(r['metrics']['timed_loop_s'] for r in rows),
    }
ab_summary = {label: summarize_current(rows) for label, rows in ab_runs.items()}
assert all(len(rows) == 2 * ABBA_ROUNDS for rows in ab_runs.values())
ab_comparison = {
    'kind': 'v5r1_827_vs_selected_abba_not_official_admission',
    'previous_commit': PREVIOUS_MEREI, 'candidate_commit': CANDIDATE,
    'memory_ceiling_mib': MEMORY_CEILING_MIB,
    'abba_rounds': ABBA_ROUNDS, 'runs_per_variant': 2 * ABBA_ROUNDS,
    'order': [{'block': index // 4 + 1, 'label': label, 'run': name}
              for index, (label, name) in enumerate(order)], 'summary': ab_summary,
}
for label, metric in [('whole_command', 'wall_median_s'), ('training_loop', 'train_loop_median_s'),
                      ('timed_steps_3_to_64', 'timed_steps_3_to_64_median_s')]:
    ab_comparison[label + '_time_reduction_pct'] = 100. * (
        1. - ab_summary['candidate'][metric] / ab_summary['previous'][metric])
ab_comparison['actual_steps_faster'] = ab_comparison['timed_steps_3_to_64_time_reduction_pct'] > 0
ab_comparison['candidate_memory_within_ceiling'] = ab_summary['candidate']['gpu_peak_mib'] <= MEMORY_CEILING_MIB
ab_comparison['candidate_memory_not_above_current'] = (
    ab_summary['candidate']['gpu_peak_mib'] <= ab_summary['previous']['gpu_peak_mib'])
(EVIDENCE / 'previous-merei-ab.json').write_text(json.dumps(ab_comparison, indent=2), encoding='utf-8')
print(json.dumps(ab_comparison, indent=2), flush=True)
assert ab_comparison['candidate_memory_within_ceiling'], 'A/B candidate exceeds 827 MiB'
assert ab_comparison['candidate_memory_not_above_current'], 'A/B candidate increases current memory'


## Отдельный official admission: два повтора, десять правил

Исходный harness выполняет base → solution → solution → base, eval и
проверки Soup outputs. `check_rules.py` обязан подтвердить все десять
правил, включая 79 регрессионных тестов. Пик каждого solution run должен
быть ≤827 MiB. Эта серия не объединяется с восемью A/B запусками.


In [ ]:
ADMISSION = 'runs_reproduce_memory827_admission'
run_repo('candidate', ['harness/run_task.py', '--repeats', '2', '--out', ADMISSION],
         'official-training.log')
run_repo('candidate', ['harness/check_rules.py', '--result', ADMISSION + '/result.json'],
         'official-checks.log')
result_path = roots['candidate'] / ADMISSION / 'result.json'
checks_path = roots['candidate'] / ADMISSION / 'checks.json'
result = json.loads(result_path.read_text())
checks = json.loads(checks_path.read_text())
assert len(checks['checks']) == 10 and all(item['ok'] for item in checks['checks'])
assert checks['result_sha256'] == hashlib.sha256(result_path.read_bytes()).hexdigest()
assert checks['solution_sha256'] == solution_hash(roots['candidate'])
assert checks['solution_sha256'] == provenance['candidate_solution_sha256_before_tests']
candidate = result['summary']['solution']
assert candidate['gpu_peak_mib'] <= MEMORY_CEILING_MIB, 'Memory827 memory ceiling exceeded'
shutil.copy2(result_path, EVIDENCE / 'final-result.json')
shutil.copy2(checks_path, EVIDENCE / 'final-checks.json')
print('ADMISSION_COMPLETE', json.dumps({
    'candidate_runtime_commit': CANDIDATE, 'candidate_summary': candidate,
    'checks_passed': 10, 'memory_ceiling_mib': MEMORY_CEILING_MIB,
}, indent=2), flush=True)

selected_runs = result['runs']['solution']
assert len(selected_runs) == 2
assert all(r['exit_code'] == 0 and r['global_step'] == 64 and r['metrics']['steps'] == 64
           and r['metrics']['timed_steps'] == 62 for r in selected_runs)
assert all(r['gpu_peak_mib'] <= MEMORY_CEILING_MIB for r in selected_runs)
assert result['packages_changed'] == []
(EVIDENCE / 'official-selected-summary.json').write_text(json.dumps({
    'candidate_commit': CANDIDATE, 'series': 'official_admission_separate_from_current_ab',
    'summary': result['summary']['solution'], 'step_summary': summarize_current(selected_runs),
    'checks_passed': 10}, indent=2), encoding='utf-8')


## Evidence и итог проверки

Сохраняются точные source blobs трёх закреплённых решений, package list,
тесты, данные, реальные init-parity captures, восемь A/B результатов и
original official result/checks. Метрики не заменяются и не корректируются.
Адаптеры и compiler caches не входят в компактный ZIP; исходные run folders
остаются в Kaggle output. Manifest содержит SHA256 каждого файла.

Требования: выигрыш времени шагов относительно V5r1, память ≤827 MiB,
неизменные source/packages и 10/10 проверок. При отрицательном результате
evidence сохраняется, затем ячейка завершает выполнение с ошибкой.


In [ ]:
def collect_small_run_artifacts(label, run_folder, relative):
    source = roots[label] / run_folder
    target = EVIDENCE / 'runs' / relative
    allowed = {'metrics.json', 'task.yaml', 'train.log', 'warmup.log',
               'trainer_state.json', 'rewind.jsonl', 'adapter_config.json',
               'config.json', 'training_args.json'}
    for path in sorted(source.rglob('*')):
        if not path.is_file() or any(p in ('triton_cache', 'inductor_cache') for p in path.parts):
            continue
        if path.name in allowed or (path.parent == source and path.suffix in ('.json', '.log')):
            destination = target / path.relative_to(source)
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(path, destination)

for label, name in order:
    collect_small_run_artifacts(label, 'runs_memory827_previous_ab/' + name, 'ab/' + name)
collect_small_run_artifacts('candidate', ADMISSION, 'official')
for label, commit in SOURCE_COMMITS.items():
    actual = verify_solution_git_blobs(roots[label], commit)
    assert actual == source_proofs[label]
    subprocess.run(['git', '-C', str(roots[label].parent), 'diff', '--exit-code', '--',
                    'Мерей/solution', 'Мерей/harness', 'Мерей/task', 'Мерей/tests'], check=True)
    for record in actual['files']:
        source = roots[label] / 'solution' / record['path']
        destination = EVIDENCE / 'sources' / label / 'solution' / record['path']
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, destination)
assert subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True) == PACKAGES_FIXED
acceptance = {
    'candidate_commit': CANDIDATE, 'previous_merei_commit': PREVIOUS_MEREI,
    'fast_reference_commit': FAST_REFERENCE,
    'anais_source_commit': ANAIS_SOURCE,
    'abba_rounds': ABBA_ROUNDS, 'ab_runs_per_variant': 2 * ABBA_ROUNDS,
    'actual_steps_faster_in_current_ab': ab_comparison['actual_steps_faster'],
    'current_ab_timed_steps_3_to_64_reduction_pct': ab_comparison['timed_steps_3_to_64_time_reduction_pct'],
    'current_ab_candidate_peak_mib': ab_summary['candidate']['gpu_peak_mib'],
    'official_candidate_peak_mib': result['summary']['solution']['gpu_peak_mib'],
    'memory_ceiling_mib': MEMORY_CEILING_MIB,
    'current_ab_memory_not_above_previous': ab_comparison['candidate_memory_not_above_current'],
    'official_checks_passed': sum(item['ok'] for item in checks['checks']),
    'solution_sha256': solution_hash(roots['candidate']),
    'solution_git_tree': source_proofs['candidate']['solution_git_tree'],
    'initialization_parity': initial_parity['all_equal'],
    'source_unchanged': True, 'packages_unchanged': True,
}
acceptance['accepted'] = (
    acceptance['actual_steps_faster_in_current_ab']
    and CANDIDATE != PREVIOUS_MEREI
    and acceptance['current_ab_memory_not_above_previous']
    and acceptance['current_ab_candidate_peak_mib'] <= MEMORY_CEILING_MIB
    and acceptance['official_candidate_peak_mib'] <= MEMORY_CEILING_MIB
    and acceptance['official_checks_passed'] == 10
    and acceptance['initialization_parity'])
(EVIDENCE / 'acceptance.json').write_text(json.dumps(acceptance, indent=2), encoding='utf-8')
manifest = {'kind': 'sha256_exact_evidence_bytes', 'files': []}
for path in sorted(EVIDENCE.rglob('*')):
    if path.is_file() and path.name != 'manifest.json':
        raw = path.read_bytes()
        manifest['files'].append({'path': path.relative_to(EVIDENCE).as_posix(), 'bytes': len(raw),
                                  'sha256': hashlib.sha256(raw).hexdigest()})
(EVIDENCE / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
for item in manifest['files']:
    raw = (EVIDENCE / item['path']).read_bytes()
    assert len(raw) == item['bytes'] and hashlib.sha256(raw).hexdigest() == item['sha256']
archive = Path(shutil.make_archive(str(EVIDENCE), 'zip', root_dir=EVIDENCE))
with zipfile.ZipFile(archive) as check_zip:
    assert check_zip.testzip() is None
    expected_members = {item['path'] for item in manifest['files']} | {'manifest.json'}
    actual_members = [item.filename for item in check_zip.infolist() if not item.is_dir()]
    assert set(actual_members) == expected_members and len(actual_members) == len(expected_members)
    for item in manifest['files']:
        content = check_zip.read(item['path'])
        assert len(content) == item['bytes'] and hashlib.sha256(content).hexdigest() == item['sha256']
receipt = {'archive': str(archive), 'bytes': archive.stat().st_size,
           'sha256': hashlib.sha256(archive.read_bytes()).hexdigest(), 'acceptance': acceptance}
EVIDENCE.with_name(EVIDENCE.name + '-receipt.json').write_text(json.dumps(receipt, indent=2), encoding='utf-8')
print(json.dumps(receipt, indent=2), flush=True)
print('EVIDENCE_DIRECTORY', str(EVIDENCE), flush=True)
assert acceptance['accepted'], 'Evidence saved; current A/B does not establish the required step improvement'
print('MEREY_STAGE3_REPRODUCTION_COMPLETE', flush=True)
